In [ ]:
import pandas as pd
import plotly.express as px
import os

data_dir = '/content/'

# Load Datasets
train_ridership = pd.read_csv(os.path.join(data_dir, "Train_Ridership_2022_to_2025H1.csv"))
shock_ridership = pd.read_csv(os.path.join(data_dir, "Shock_Ridership_2025_Q3.csv"))
train_traffic = pd.read_csv(os.path.join(data_dir, "Train_Traffic_2022_to_2025H1.csv"))
shock_traffic = pd.read_csv(os.path.join(data_dir, "Shock_Traffic_2025_Q3.csv"))

# Convert dates and label periods
train_ridership['Date'] = pd.to_datetime(train_ridership['Date'])
shock_ridership['Date'] = pd.to_datetime(shock_ridership['Date'])
train_traffic['Date'] = pd.to_datetime(train_traffic['Date'])
shock_traffic['Date'] = pd.to_datetime(shock_traffic['Date'])

train_ridership['Period'] = 'Historical (2022-2025H1)'
shock_ridership['Period'] = 'Shock (2025 Q3)'
train_traffic['Period'] = 'Historical (2022-2025H1)'
shock_traffic['Period'] = 'Shock (2025 Q3)'

# Combine data
traffic_all = pd.concat([train_traffic, shock_traffic]).sort_values('Date')
ridership_all = pd.concat([train_ridership, shock_ridership])

# --- Chart 1: Traffic Congestion ---
# Group by Month-Year for smoother trends across the entire timeline
traffic_all['Month'] = traffic_all['Date'].dt.to_period('M').dt.to_timestamp()
monthly_traffic = traffic_all.groupby(['Month', 'Period']).agg({'Congestion_Level': 'mean', 'Avg_Speed_kmph': 'mean'}).reset_index()

fig_congestion = px.line(monthly_traffic, x='Month', y='Congestion_Level', color='Period',
                         title='Average Monthly Congestion Level over Time (Historical vs Shock)', markers=True)
fig_congestion.show()

# --- Chart 2: Daily Ridership (Zoomed into 2025) ---
daily_ridership = ridership_all.groupby(['Date', 'Period'])['Boarding_Count'].sum().reset_index()
ridership_2025 = daily_ridership[daily_ridership['Date'].dt.year == 2025]

fig_ridership = px.line(ridership_2025, x='Date', y='Boarding_Count', color='Period',
                        title='Daily Total Boardings in 2025 (Showing the Shock)', markers=True)
fig_ridership.show()

In [ ]:
import pandas as pd
import plotly.express as px
import os


# Load Datasets
train_ridership = pd.read_csv(os.path.join(data_dir, "Train_Ridership_2022_to_2025H1.csv"))
shock_ridership = pd.read_csv(os.path.join(data_dir, "Shock_Ridership_2025_Q3.csv"))
bus_routes = pd.read_csv(os.path.join(data_dir, "Bus_Routes.csv"))
bus_stops = pd.read_csv(os.path.join(data_dir, "Bus_Stops.csv"))

# Convert dates
train_ridership['Date'] = pd.to_datetime(train_ridership['Date'])
shock_ridership['Date'] = pd.to_datetime(shock_ridership['Date'])

# Merge Route Types and Zones into the data
train_ridership = train_ridership.merge(bus_routes[['Route_ID', 'Route_Type', 'Route_Code']], on='Route_ID', how='left')

# Fair baseline: We compare only H1 2025 versus the Shock in Q3 2025
train_ridership_2025 = train_ridership[train_ridership['Date'].dt.year == 2025].copy()
train_ridership_2025['Period'] = 'Historical (H1 2025)'
shock_ridership['Period'] = 'Shock (Q3 2025)'

# Combine test vs train
ridership = pd.concat([train_ridership_2025, shock_ridership])
ridership['Total_Pax'] = ridership['Boarding_Count'] + ridership['Alighting_Count']

# --- Chart 3: Impact by Route Type (Bar Chart) ---
route_type_daily = ridership.groupby(['Date', 'Route_Type', 'Period'])['Total_Pax'].sum().reset_index()
route_type_summary = route_type_daily.groupby(['Route_Type', 'Period'])['Total_Pax'].mean().reset_index()

fig_routes = px.bar(route_type_summary, x='Route_Type', y='Total_Pax', color='Period', barmode='group',
                    title='Average Daily Passengers by Route Type (H1 2025 vs Shock)',
                    labels={'Total_Pax': 'Avg Daily Passengers'})
fig_routes.show()

# --- Chart 4: Impact by Zone (Bar Chart) ---
ridership = ridership.merge(bus_stops[['Stop_ID', 'Zone']], on='Stop_ID', how='left')
zone_daily = ridership.groupby(['Date', 'Zone', 'Period'])['Total_Pax'].sum().reset_index()
zone_summary = zone_daily.groupby(['Zone', 'Period'])['Total_Pax'].mean().reset_index()

fig_zones = px.bar(zone_summary, x='Zone', y='Total_Pax', color='Period', barmode='group',
                   title='Average Daily Passengers by Zone (H1 2025 vs Shock)',
                   labels={'Total_Pax': 'Avg Daily Passengers'})
fig_zones.show()

# --- Chart 5: Top 10 Most Impacted Routes (Heatmap / Color-Scaled Bar) ---
route_daily = ridership.groupby(['Date', 'Route_Code', 'Period'])['Total_Pax'].sum().reset_index()
route_summary = route_daily.groupby(['Route_Code', 'Period'])['Total_Pax'].mean().unstack().reset_index()
route_summary['% Change'] = (route_summary['Shock (Q3 2025)'] - route_summary['Historical (H1 2025)']) / route_summary['Historical (H1 2025)'] * 100
route_summary = route_summary.sort_values('% Change', ascending=False).head(10)

fig_top10 = px.bar(route_summary, x='Route_Code', y='% Change',
                   title='Top 10 Routes with Highest Percentage Increase During Shock',
                   labels={'% Change': 'Passenger Increase (%)'},
                   color='% Change', color_continuous_scale='Reds')
fig_top10.show()
